In [19]:
import pandas as pd
import numpy as np
import geopandas as gpd

import folium
from folium import Choropleth
import openpyxl

The objective of this file is to calculate the proportion between the visits that a town receives and its population. It's a first step for town "attraction" calculation.

In [20]:
file_path = "data/data_oct_2024.csv"
viajes_completo = pd.read_csv(file_path, sep=',')
filtro_viajes = viajes_completo['origen'].str.startswith('20') & viajes_completo['destino'].str.startswith('20')
viajes_completo = viajes_completo[filtro_viajes]
viajes_completo = viajes_completo.reset_index(drop=True)
viajes_completo.head()

,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,20241001,13,20005_AM,20005_AM,0.5-2,frecuente,casa,no,no,20,>15,NaN,NaN,2.429,4.749
1,20241001,15,20005_AM,20005_AM,0.5-2,frecuente,casa,no,no,20,>15,NaN,NaN,3.425,3.169
2,20241001,23,20005_AM,20005_AM,0.5-2,frecuente,casa,no,no,20,>15,NaN,NaN,2.429,4.749
3,20241001,12,20019,20005_AM,0.5-2,frecuente,casa,no,no,20,>15,NaN,NaN,2.364,4.594
4,20241001,18,20019,20005_AM,0.5-2,frecuente,casa,no,no,20,>15,NaN,NaN,2.364,3.692


# Analysis

In [21]:
# I select the destination as not HOME or WORK

filtro_viajes = viajes_completo['destino'].str.startswith('20')
viajes_completo = viajes_completo[filtro_viajes]
dnf = viajes_completo[(viajes_completo['actividad_destino'] == 'no_frecuente') | (viajes_completo['actividad_destino'] == 'frecuente')]

dnf['viajes'] = dnf['viajes'].apply(lambda x: int(x + 0.5) if x >= 0 else int(x - 0.5))

analysis = dnf.drop(columns=['actividad_origen', 'actividad_destino', 'distancia', 'estudio_origen_posible', 'estudio_destino_posible', 'residencia', 'renta', 'edad', 'sexo', 'viajes_km'])

C:\Users\iazcarateu\AppData\Local\Temp\ipykernel_14232\1159336613.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dnf['viajes'] = dnf['viajes'].apply(lambda x: int(x + 0.5) if x >= 0 else int(x - 0.5))


In [22]:
# Grouping by fecha, periodo and destino

analysis_sum = analysis.copy()
analysis_sum = analysis_sum.drop(columns='origen')

new_df = analysis_sum.groupby(['fecha', 'periodo', 'destino'], as_index=False).agg({'viajes': 'sum'})

# Renaming the 'viajes' column to 'total'
new_df.rename(columns={'viajes': 'total'}, inplace=True)
new_df

,fecha,periodo,destino,total
0,20241001,0,20005_AM,21
1,20241001,0,20009,250
2,20241001,0,20010_AM,26
3,20241001,0,20013,10
4,20241001,0,20016_AM,74
...,...,...,...,...
50565,20241031,23,2007902,418
50566,20241031,23,20080,165
50567,20241031,23,20081,163
50568,20241031,23,20902,271


In [23]:
# I would do the media, but for the sake of simplicity I am going to take two random days, one week and one weekend.
# At the end, just one day, Tuesday 15th of October

selected = new_df[new_df['fecha'] == 20241015].reset_index(drop=True)
selected = selected.drop(columns='fecha')

# Group discriminating the hours

selected = selected.groupby(['destino'], as_index=False).agg({'total': 'sum'})

# Bring the shapefile

In [24]:
distritos = gpd.read_file('data/zonificacion_distritos/zonificacion_distritos.shp')
centroides = gpd.read_file('data/zonificacion_distritos_centroides/zonificacion_distritos_centroides.shp')

filtro_distritos = distritos['ID'].str.startswith('20')
distritos = distritos[filtro_distritos]

filtro_centroides = centroides['ID'].str.startswith('20')
centroides = centroides[filtro_centroides]

c:\Users\iazcarateu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: data/zonificacion_distritos/zonificacion_distritos.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [25]:
# Merge dfs

merged_df = pd.merge(selected, distritos, left_on='destino', right_on='ID', how='inner')
merged_df = merged_df.drop(columns='ID')
merged_df = pd.merge(merged_df, centroides, left_on='destino', right_on='ID', how='inner')
merged_df = merged_df.drop(columns='ID')
merged_df = merged_df.rename(columns={'geometry_x': 'geometry', 'geometry_y': 'centroide'})
merged_df

,destino,total,geometry,centroide
0,20005_AM,3524,"MULTIPOLYGON (((570773.179 4760183.509, 570764...",POINT (573490.172 4765089.042)
1,20009,15190,"POLYGON ((578805.914 4788107.614, 578824.349 4...",POINT (580146.768 4785278.111)
2,20010_AM,5022,"POLYGON ((575438.532 4777955.062, 575436.871 4...",POINT (573398.683 4779697.176)
3,20013,3801,"POLYGON ((542400.685 4757160.559, 542203.185 4...",POINT (541922.531 4763016.126)
4,20016_AM,6078,"POLYGON ((574024.213 4781912.807, 574020.338 4...",POINT (569823.639 4786686.907)
...,...,...,...,...
63,2007902,16734,"MULTIPOLYGON (((565122.897 4789724.538, 564661...",POINT (567415.976 4791529.195)
64,20080,9045,"POLYGON ((555035.836 4769953.378, 555023.145 4...",POINT (556149.05 4772383.095)
65,20081,7990,"POLYGON ((560602.497 4793752.154, 560681.374 4...",POINT (561303.968 4792783.209)
66,20902,15258,"POLYGON ((580606.092 4788608.309, 580375.355 4...",POINT (580133.549 4790186.703)


In [26]:
merged_df.to_csv('merged_df.csv')

# Add data

In [27]:
merged_df = pd.read_csv('merged_df.csv', index_col=False)
merged_df = merged_df.drop(columns=['Unnamed: 0', 'centroide'])
merged_df

,destino,total,geometry
0,20005_AM,3524,MULTIPOLYGON (((570773.1789999995 4760183.5088...
1,20009,15190,"POLYGON ((578805.9143000003 4788107.613700001,..."
2,20010_AM,5022,"POLYGON ((575438.5317000002 4777955.0624, 5754..."
3,20013,3801,POLYGON ((542400.6846000003 4757160.5594999995...
4,20016_AM,6078,"POLYGON ((574024.2131000003 4781912.806600001,..."
...,...,...,...
63,2007902,16734,MULTIPOLYGON (((565122.8971999995 4789724.5380...
64,20080,9045,"POLYGON ((555035.8364000004 4769953.3781, 5550..."
65,20081,7990,"POLYGON ((560602.4968999997 4793752.1535, 5606..."
66,20902,15258,"POLYGON ((580606.0915999999 4788608.308599999,..."


In [28]:
poblacion = pd.read_csv('data/poblacion_distritos.csv', sep='|')
poblacion = poblacion[poblacion['ID'].str.startswith('20')]

merged_df = pd.merge(merged_df, poblacion, left_on='destino', right_on='ID', how='left')
merged_df = merged_df.drop(columns=['ID'])
merged_df['vis/pob'] = merged_df['total'] / merged_df['population']

In [29]:
# Bring other town data

data_towns = pd.read_excel('data/data_towns.xlsx')
data_towns = data_towns.drop(columns=['Region', 'Latitud', 'Longitud', 'Comarca', 'Altitud (m.s.n.m.)', 'Superficie (kmÂ²)', 'PoblaciÃ³n (2019)', 'Código', 'Incluido'])
data_towns = data_towns.rename(columns={'Densidad (hab./kmÂ²)': 'Density'})

ine_name = pd.read_excel('data/26codmun.xlsx', sheet_name='20', skiprows=2)
ine_name['CPRO'] = ine_name['CPRO'].astype(str)
ine_name['CMUN'] = ine_name['CMUN'].astype(str)
ine_name['DC'] = ine_name['DC'].astype(str)
ine_name['CMUN'] = ine_name['CMUN'].str.zfill(3)
ine_name['CODE'] = ine_name['CPRO'].astype(str) + ine_name['CMUN'].astype(str)
ine_name = ine_name.drop(columns=['CPRO', 'CMUN', 'DC'])
ine_name['CODE'] = ine_name['CODE'].astype(str)

mitma_ine = pd.read_csv('data/relacion_ine_zonificacionMitma.csv', sep='|')
mitma_ine = mitma_ine[mitma_ine['distrito_mitma'].str.startswith('20')]
mitma_ine = mitma_ine.drop(columns=['seccion_ine', 'distrito_ine', 'municipio_mitma', 'gau_mitma'])
mitma_ine['municipio_ine'] = mitma_ine['municipio_ine'].astype(int)
mitma_ine['municipio_ine'] = mitma_ine['municipio_ine'].astype(str)

In [30]:
merge_1 = pd.merge(mitma_ine, ine_name, left_on='municipio_ine', right_on='CODE', how='inner')
merge_1 = merge_1.drop(columns='municipio_ine')
merge_1 = merge_1.drop_duplicates()
merge_2 = pd.merge(merge_1,data_towns, left_on='NOMBRE', right_on='Town', how='inner')
merge_2 = merge_2.drop(columns=['NOMBRE', 'CODE'])

prueba = pd.merge(merged_df,merge_2, left_on='destino', right_on='distrito_mitma', how='inner')

# I will take from the same mitma district, the town/row with the highest density
prueba = prueba.loc[prueba.groupby('destino')['Density'].idxmax()]
prueba = prueba.drop(columns=['distrito_mitma'])
prueba.head(10)

,destino,total,geometry,population,vis/pob,Town,Density
1,20005_AM,3524,MULTIPOLYGON (((570773.1789999995 4760183.5088...,5541.0,0.635986,Alegia,224.22
9,20009,15190,"POLYGON ((578805.9143000003 4788107.613700001,...",14631.0,1.038207,Andoain,538.72
13,20010_AM,5022,"POLYGON ((575438.5317000002 4777955.0624, 5754...",4646.0,1.080930,Irura,619.40
14,20013,3801,POLYGON ((542400.6846000003 4757160.5594999995...,7108.0,0.534750,Aretxabaleta,244.70
15,20016_AM,6078,"POLYGON ((574024.2131000003 4781912.806600001,...",3864.0,1.572981,Asteasu,92.54
18,2001701,2549,"POLYGON ((555849.0864000004 4780615.852, 55581...",2778.0,0.917567,Azkoitia,212.63
19,2001702,2712,"POLYGON ((555983.6524 4780795.3467999995, 5559...",3356.0,0.808105,Azkoitia,212.63
20,2001703,4806,"POLYGON ((556488.8355 4780431.834899999, 55646...",5523.0,0.870179,Azkoitia,212.63
21,20018,9347,"POLYGON ((562451.3685999997 4776308.1719, 5624...",15191.0,0.615299,Azpeitia,215.25
22,20019,23719,"POLYGON ((565842.7879999997 4766384.601199999,...",13949.0,1.700409,Beasain,462.82


In [31]:
# establecimientos

esta = pd.read_excel('data/establecimientos.xlsx')
towns = pd.read_excel('data/data_towns.xlsx')
towns = towns[['Town', 'Latitud', 'Longitud']]
esta = pd.merge(esta, towns, left_on='municipio', right_on='Town')
esta = esta.drop(columns='Town')

In [32]:
from geopy.distance import geodesic

def calcular_distancias(row, df):
    origen = (row['Latitud'], row['Longitud'])
    df['distancia'] = df.apply(lambda x: geodesic(origen, (x['Latitud'], x['Longitud'])).kilometers, axis=1)
    return df.sort_values('distancia').iloc[1:11]  # Selecciona los 10 más cercanos (excluyendo el propio pueblo)

# Lista para almacenar las medias de distancia
medias_distancia = []

# Iterar sobre los pueblos
for index, row in esta.iterrows():
    pueblos_cercanos = calcular_distancias(row, esta)
    media = pueblos_cercanos['distancia'].mean()
    medias_distancia.append(media)

# Agregar la nueva columna al DataFrame
esta['connectivity'] = medias_distancia
esta['connectivity'] = esta['connectivity'].round(1)

esta = esta.rename(columns={'total': 'Establishments', 'connectivity': 'Town connectivity'})
esta = esta.drop(columns=['Longitud', 'Latitud', 'distancia'])

In [33]:
prueba = pd.merge(prueba, esta, left_on='Town', right_on='municipio', how='inner')
prueba.head(10)

,destino,total,geometry,population,vis/pob,Town,Density,municipio,Establishments,Town connectivity
0,20005_AM,3524,MULTIPOLYGON (((570773.1789999995 4760183.5088...,5541.0,0.635986,Alegia,224.22,Alegia,120,4.0
1,20009,15190,"POLYGON ((578805.9143000003 4788107.613700001,...",14631.0,1.038207,Andoain,538.72,Andoain,886,5.4
2,20010_AM,5022,"POLYGON ((575438.5317000002 4777955.0624, 5754...",4646.0,1.080930,Irura,619.40,Irura,147,3.7
3,20013,3801,POLYGON ((542400.6846000003 4757160.5594999995...,7108.0,0.534750,Aretxabaleta,244.70,Aretxabaleta,355,10.7
4,20016_AM,6078,"POLYGON ((574024.2131000003 4781912.806600001,...",3864.0,1.572981,Asteasu,92.54,Asteasu,183,4.7
5,2001701,2549,"POLYGON ((555849.0864000004 4780615.852, 55581...",2778.0,0.917567,Azkoitia,212.63,Azkoitia,675,8.9
6,2001702,2712,"POLYGON ((555983.6524 4780795.3467999995, 5559...",3356.0,0.808105,Azkoitia,212.63,Azkoitia,675,8.9
7,2001703,4806,"POLYGON ((556488.8355 4780431.834899999, 55646...",5523.0,0.870179,Azkoitia,212.63,Azkoitia,675,8.9
8,20018,9347,"POLYGON ((562451.3685999997 4776308.1719, 5624...",15191.0,0.615299,Azpeitia,215.25,Azpeitia,1194,8.5
9,20019,23719,"POLYGON ((565842.7879999997 4766384.601199999,...",13949.0,1.700409,Beasain,462.82,Beasain,991,5.0
